<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Evaluate standalone policy-target tests (cost effectiveness, CBA, distributional impact).
- **Primary inputs**: runs/policies_test/assessment_test_*/policies/indicator.csv + data/policies_insulation_description.csv
- **Primary outputs**: diagnostic scatter plots and test.csv extracts
- **Refactor role**: Keep for targeted policy testing; use shared indicator parser and standardized scenario metadata loading.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Load policy test indicators for one assessment test run.
2. Join scenario metadata describing target/index/value combinations.
3. Generate scatter plots for cost-effectiveness, CBA, carbon, and distributional dimensions.

### Inputs
- `runs/policies_test/assessment_test_<date_time>/policies/indicator.csv`
- `data/policies_insulation_description.csv`

### Outputs
- Diagnostic scatter plots displayed in notebook.
- Optional exported test table in `data/test_policy_targets.csv`.

### How To Reuse
- Update `folder` to the policy test run id you want.


# Test impact of individual policy based on their targets.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from project.utils import format_ax

In [ ]:
# Refactor: policy-target test runs are under policy_assessment/runs/policies_test
folder = os.path.join('runs', 'policies_test', 'assessment_test_20230717_142815')

In [ ]:
# Read scenarios result
data = pd.read_csv(os.path.join(folder, 'policies/indicator.csv'), index_col=[0])
data.columns = data.columns.str.replace('ZP+', '', regex=False)
data = data.stack().rename_axis(['Variables', 'Scenarios'], axis=0)

# Read scenarios description
scenarios = pd.read_csv(os.path.join('data', 'policies_insulation_description.csv'), index_col=[0]).rename_axis('Scenarios')

data = data.to_frame().join(scenarios, on='Scenarios').set_index(list(scenarios.columns), append=True).squeeze()
data = data.unstack('Variables')
data = data.reset_index()
data["target"].fillna("no target", inplace=True)
data["index"].fillna("no target", inplace=True)
data["index"] = data["index"].str.replace('project/input/policies/target/low_income_owner.csv', 'Low income owner', regex=False)

## Cost effectiveness energy

In [ ]:
fig, ax = plt.subplots(figsize=(12.8, 9.6))

sns.scatterplot(
    data=data.loc[data["value"] == 0.3, :], x="Consumption saving (TWh/year)", y="Cost effectiveness (euro/kWh)", hue="target",
    style="index", ax=ax, s=200
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

format_ax(ax, title='Cost effectiveness (euro per kWh)', format_y=lambda y, _: '{:.3f}'.format(y), y_label='',
          format_x=lambda y, _: '{:.1f}'.format(y))

## Cost policies

In [ ]:
condition = (data["value"] == 0.3) & (data["index"] == "no target")

fig, ax = plt.subplots(figsize=(12.8, 9.6))

sns.scatterplot(
    data=data.loc[condition, :], x="Consumption saving (TWh/year)", y="Policy test (Billion euro)", hue="target",
    style="target", ax=ax, s=200
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

format_ax(ax, title='Policies cost (Billion euro)', format_y=lambda y, _: '{:.3f}'.format(y), y_label='',
          format_x=lambda y, _: '{:.1f}'.format(y))

## Cost-benefits analysis

In [ ]:
condition = (data["value"] == 0.3) & (data["index"] == "no target")

fig, ax = plt.subplots(figsize=(12.8, 9.6))

sns.scatterplot(
    data=data.loc[condition, :], x="Consumption saving (TWh/year)", y="Cost-benefits analysis (Billion euro/year)", hue="target",
    style="target", ax=ax, s=200
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

format_ax(ax, title='Cost-benefits analysis (Billion euro per year)', format_y=lambda y, _: '{:.3f}'.format(y), y_label='',
          format_x=lambda y, _: '{:.1f}'.format(y))

## Cost effectiveness carbon

In [ ]:
condition = (data["value"] == 0.3) & (data["index"] == "no target")

fig, ax = plt.subplots(figsize=(12.8, 9.6))

sns.scatterplot(
    data=data.loc[condition, :], x="Emission saving (MtCO2/year)", y="Cost effectiveness carbon (euro/tCO2)", hue="target",
    style="target", ax=ax, s=200
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

format_ax(ax, title='Cost effectiveness carbon (euro per tCO2)', format_y=lambda y, _: '{:.0f}'.format(y), y_label='',
          format_x=lambda y, _: '{:.2f}'.format(y))

## Distributive impact

In [ ]:
condition = (data["value"] == 0.3)

fig, ax = plt.subplots(figsize=(12.8, 9.6))

sns.scatterplot(
    data=data.loc[condition, :], x="Consumption saving (TWh/year)", y='Balance Owner-occupied - C1 (euro/year.household)', hue="target",
    style="index", ax=ax, s=200
)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

format_ax(ax, title='Balance Owner-occupied - C1 (euro/year.household)', format_y=lambda y, _: '{:.0f}'.format(y), y_label='',
          format_x=lambda y, _: '{:.2f}'.format(y))